In [ ]:
import email
import imaplib
import re
from datetime import datetime
from io import BytesIO

from os.path import join

from pathlib import Path

import camelot
import pandas as pd
from bs4 import BeautifulSoup

from glob import glob

In [2]:
prev = 0


def pretty(label: str, completed: float, total: int, length: int = 30) -> None:
    global prev
    print(
        " " * (prev * 2)
        + f"\r{label} ["
        + "=" * int(completed / total * length)
        + "-" * int((total - completed) / total * length)
        + f"] {int(completed / total * 100)}%",
        end="",
    )
    prev = length


def prettyPrintReplace(label: str) -> None:
    global prev
    label = str(label)
    print((" " * (prev * 2)) + f"\r{label}", end="")
    prev = len(label)

In [3]:
# Login credentials
IMAP_HOST = "imap.gmail.com"
EMAIL_USER = "ankushmisal7387@gmail.com"
EMAIL_PASS = "lnfv cnrn yrju ukom"

In [4]:
subject_to_search = "UPGRADE MANDI - Hyperpure"

In [5]:
# Connect and login
mail = imaplib.IMAP4_SSL(IMAP_HOST)
mail.login(EMAIL_USER, EMAIL_PASS)
mail.select("inbox")

statusRaw, messagesRaw = mail.search(None, f'(SUBJECT "{subject_to_search}")')

In [6]:
grnPrDF = pd.DataFrame(
    {"Cut/Veg": [], "Date": [], "GRN": [], "PO Amount": [], "Error": []}
)

status = statusRaw
messages = messagesRaw
total_mails = len(messages[0].split())
if status == "OK":
    for num in messages[0].split():
        typ, data = mail.fetch(num, "(RFC822)")
        msg = email.message_from_bytes(data[0][1])
        sub = msg["Subject"]
        prettyPrintReplace(f"Count: {len(grnPrDF) + 1}/{total_mails} {sub}")

        processedDOMTree = ""
        cutVeg = {"value": None}

        subject = re.sub("\s+", " ", msg["subject"])
        indexPO = re.search(r"CPCMH\d{2}-PO-\d{7}", subject)

        isGrnAmount = "UPGRADE MANDI - Hyperpure GRN against PO Number" in subject
        # print(isGrnAmount)
        isTotalAmount = "UPGRADE MANDI - Hyperpure PO Number" in subject

        if indexPO:
            indexPO = indexPO.group()
        else:
            raise Exception("Can't find PO number from mail.")

        if indexPO not in grnPrDF.index:
            grnPrDF.loc[indexPO] = ["", "", 0.0, 0.0, ""]

        for part in msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition") or "")

            if part.get_content_maintype() == "multipart":
                continue
            elif content_type == "application/pdf":
                # cutVeg = "VEG"

                pdfFileData = part.get_payload(decode=True)
                pdfFileStream = BytesIO(pdfFileData)

                # print(sub.split(" ")[-1])

                try:
                    with open(
                        join(
                            "dump", f"{indexPO} - {'GRN' if isGrnAmount else 'PO'}.pdf"
                        ),
                        "wb",
                    ) as f:
                        f.write(pdfFileStream.getbuffer())
                except IOError:
                    cutVeg = "Error"
                except Exception as e:
                    print("Other error")
                    print(e)
                    print(sub)

                # grnPrDF.loc[indexPO, "Cut/Veg"] = cutVeg
            elif content_type == "text/html":
                rawHTMLText = part.get_payload(decode=True).decode()

                processedDOMTree = BeautifulSoup(rawHTMLText, "html.parser")
                try:
                    amountDate = datetime.strptime(
                        processedDOMTree.find_all("table")[1]
                        .find_all("tr")[3]
                        .find_all("td")[1]
                        .find("span")
                        .text,
                        "%d %b %Y",
                    ).strftime("%d-%m-%Y")

                    amount = float(
                        processedDOMTree.find_all("table")[1]
                        .find_all("tr")[4]
                        .find_all("td")[1]
                        .find("span")
                        .text
                    )

                    if isGrnAmount:
                        grnPrDF.loc[indexPO, "GRN"] += amount
                    if isTotalAmount:
                        grnPrDF.loc[indexPO, "PO Amount"] += amount
                    grnPrDF.loc[indexPO, "Date"] = amountDate
                except Exception as e:
                    grnPrDF.loc[indexPO, "Error"] += f"{e};;"
                    pass

        # print(
        #     f"Date: {amountDate}",
        #     f"Index PO: {indexPO}",
        #     f'GRN total: {grnPrDF.iloc[-1]["GRN"]}',
        #     f'Total: {grnPrDF.iloc[-1]["PO Amount"]}',
        #     f'Cut: {grnPrDF.iloc[-1]["Cut/Veg"]}',
        #     sep="\n",
        #     end="\n\n\n",
        # )

Count: 560/1161 UPGRADE MANDI	 - Hyperpure - Hyperpure PO Scheduled for PO Number34                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [31]:
grnPrDF.index.name = "PO Number"
grnPrDF.to_csv("./GRN PR (Blinkit).csv", index=True)

In [24]:
grnPrDF

,PO Amount,Cut/Veg,Date,GRN,PO Amount,Error
PO Number,,,,,,
0,CPCMH25-PO-1269666,NaN,05-12-2024,18788.5,18806.0,NaN
1,CPCMH25-PO-1274368,NaN,07-12-2024,3679.5,3679.5,NaN
2,CPCMH25-PO-1276694,NaN,08-12-2024,12899.5,12956.5,NaN
3,CPCMH25-PO-1279272,NaN,09-12-2024,9014.0,9028.0,NaN
4,CPCMH25-PO-1281280,NaN,10-12-2024,52301.7,55274.2,NaN
...,...,...,...,...,...,...
566,CPCMH26-PO-2244427,NaN,03-10-2025,7863.0,15268.5,list index out of range;;
567,CPCMH26-PO-2248894,NaN,04-10-2025,14854.4,26797.0,list index out of range;;
568,CPCMH26-PO-2252935,NaN,05-10-2025,17686.8,23482.2,list index out of range;;


In [34]:
grnPrDF = pd.read_csv("./GRN PR (Blinkit).csv")
grnPrDF.index = grnPrDF["PO Amount"]
grnPrDF.drop(columns=["PO Amount"], inplace=True)

In [35]:
grnPrDF

,PO Number,Cut/Veg,Date,GRN,PO Amount.1,Error
PO Amount,,,,,,
CPCMH25-PO-1269666,CPCMH25-PO-1269666,NaN,05-12-2024,18788.5,18806.0,NaN
CPCMH25-PO-1274368,CPCMH25-PO-1274368,NaN,07-12-2024,3679.5,3679.5,NaN
CPCMH25-PO-1276694,CPCMH25-PO-1276694,NaN,08-12-2024,12899.5,12956.5,NaN
CPCMH25-PO-1279272,CPCMH25-PO-1279272,NaN,09-12-2024,9014.0,9028.0,NaN
CPCMH25-PO-1281280,CPCMH25-PO-1281280,NaN,10-12-2024,52301.7,55274.2,NaN
...,...,...,...,...,...,...
CPCMH26-PO-2244427,CPCMH26-PO-2244427,NaN,03-10-2025,7863.0,15268.5,list index out of range;;
CPCMH26-PO-2248894,CPCMH26-PO-2248894,NaN,04-10-2025,14854.4,26797.0,list index out of range;;
CPCMH26-PO-2252935,CPCMH26-PO-2252935,NaN,05-10-2025,17686.8,23482.2,list index out of range;;


In [36]:
file_list = glob(join("dump", "*.pdf"))
file_list = list(filter(lambda x: ("GRN.pdf" in x) or ("PO.pdf" in x), file_list))
ln = len(file_list)
for i, path in enumerate(file_list):
    cutVeg = "VEG"
    # print(f'"{grnPrDF.loc[Path(path).stem.split(" - ")[0], "Cut/Veg"]}"')
    if str(grnPrDF.loc[Path(path).stem.split(" - ")[0], "Cut/Veg"]) != "nan":
        continue

    try:
        pdfParserObject = camelot.read_pdf(path)
        if len(pdfParserObject[0].df) >= 5:
            findCut = pdfParserObject[0].df.iloc[5][1]
            findMatches = re.findall(r".*cut.*", findCut, re.IGNORECASE)
            if len(findMatches) != 0:
                cutVeg = "CUT-VEG"
    except:
        cutVeg = "Error"
    grnPrDF.loc[Path(path).stem.split(" - ")[0], "Cut/Veg"] = cutVeg
    print(
        f"{i + 1}/{ln}, {Path(path).stem.split( ' - ')[0]}, {findMatches}, {grnPrDF.loc[Path(path).stem.split(' - ')[0], 'Cut/Veg']}"
    )
    # if i > 100:
    #     break

C:\Users\cw\AppData\Local\Temp\ipykernel_15292\2237780842.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'VEG' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grnPrDF.loc[Path(path).stem.split(" - ")[0], "Cut/Veg"] = cutVeg


1/1129, CPCMH25-PO-1269666, [], VEG
3/1129, CPCMH25-PO-1274368, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
5/1129, CPCMH25-PO-1276694, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
7/1129, CPCMH25-PO-1279272, [], VEG
9/1129, CPCMH25-PO-1281280, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
11/1129, CPCMH25-PO-1283870, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
13/1129, CPCMH25-PO-1286137, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
15/1129, CPCMH25-PO-1288444, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
17/1129, CPCMH25-PO-1290703, [], VEG
18/1129, CPCMH25-PO-1291612, [], VEG
20/1129, CPCMH25-PO-1293847, [], VEG
22/1129, CPCMH25-PO-1294697, [], VEG
24/1129, CPCMH25-PO-1295966, [], VEG
26/1129, CPCMH25-PO-1296693, [], VEG
28/1129, CPCMH25-PO-1298679, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
30/1129, CPCMH25-PO-1300959, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
32/1129, CPCMH25-PO-1302565, [], VEG
33/1129, CPCMH25-PO-1303813, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
35/1129, CPCMH25-PO-1306159, ['BH-Pumpkin Cut, 250 gm'], CUT-VEG
37/1129, CPCMH25-PO-13

EOF marker not found


643/1129, CPCMH26-PO-1804198, [], VEG
645/1129, CPCMH26-PO-1804203, [], Error
647/1129, CPCMH26-PO-1806887, [], VEG
649/1129, CPCMH26-PO-1807636, [], VEG
651/1129, CPCMH26-PO-1807644, ['BH- Green Pumpkin Cut, 250'], CUT-VEG
653/1129, CPCMH26-PO-1808108, [], VEG
655/1129, CPCMH26-PO-1810250, [], VEG
657/1129, CPCMH26-PO-1810731, [], VEG
659/1129, CPCMH26-PO-1810737, ['BH- Green Pumpkin Cut, 250', 'BH-Jackfruit (Kathal) - Cut,', 'BH-Cut Sambhar Mix, 200 gm', 'BH - Bitter Gourd - Cut, 250', 'BH-Drumstick - Cut, 200 gm'], CUT-VEG
661/1129, CPCMH26-PO-1811485, [], VEG
663/1129, CPCMH26-PO-1814172, [], VEG
665/1129, CPCMH26-PO-1814732, [], VEG
667/1129, CPCMH26-PO-1814735, ['BH-Jackfruit (Kathal) - Cut,', 'BH-Cut Sambhar Mix, 200 gm', 'BH - Bitter Gourd - Cut, 250', 'BH-Drumstick - Cut, 200 gm', 'BH- Green Pumpkin Cut, 250'], CUT-VEG
669/1129, CPCMH26-PO-1817761, [], VEG
671/1129, CPCMH26-PO-1818527, [], VEG
673/1129, CPCMH26-PO-1818534, ['BH-Jackfruit (Kathal) - Cut,', 'BH-Drumstick - Cut, 

EOF marker not found
EOF marker not found


819/1129, CPCMH26-PO-1899269, [], VEG
820/1129, CPCMH26-PO-1900179, [], Error
822/1129, CPCMH26-PO-1900186, [], Error
824/1129, CPCMH26-PO-1904132, [], VEG
826/1129, CPCMH26-PO-1904138, ['BH - Bitter Gourd - Cut, 250', 'BH-Drumstick - Cut, 200 gm', 'BH- Green Pumpkin Cut, 250', 'BH-Jackfruit (Kathal) - Cut,'], CUT-VEG
828/1129, CPCMH26-PO-1908056, [], VEG
830/1129, CPCMH26-PO-1908398, [], VEG
832/1129, CPCMH26-PO-1911819, [], VEG
834/1129, CPCMH26-PO-1911830, [], VEG
836/1129, CPCMH26-PO-1915082, [], VEG
838/1129, CPCMH26-PO-1915191, ['BH-Jackfruit (Kathal) - Cut,'], CUT-VEG
840/1129, CPCMH26-PO-1919136, [], VEG
842/1129, CPCMH26-PO-1919598, ['BH-Jackfruit (Kathal) - Cut,'], CUT-VEG
844/1129, CPCMH26-PO-1923091, [], VEG
846/1129, CPCMH26-PO-1923097, ['BH- Green Pumpkin Cut, 250', 'BH-Jackfruit (Kathal) - Cut,'], CUT-VEG
848/1129, CPCMH26-PO-1923110, ['BH - Bitter Gourd - Cut, 250', 'BH-Drumstick - Cut, 200 gm', 'BH-Cut Sambhar Mix, 200 gm'], CUT-VEG
850/1129, CPCMH26-PO-1927051, [], VE

In [37]:
grnPrDF

,PO Number,Cut/Veg,Date,GRN,PO Amount.1,Error
PO Amount,,,,,,
CPCMH25-PO-1269666,CPCMH25-PO-1269666,VEG,05-12-2024,18788.5,18806.0,NaN
CPCMH25-PO-1274368,CPCMH25-PO-1274368,CUT-VEG,07-12-2024,3679.5,3679.5,NaN
CPCMH25-PO-1276694,CPCMH25-PO-1276694,CUT-VEG,08-12-2024,12899.5,12956.5,NaN
CPCMH25-PO-1279272,CPCMH25-PO-1279272,VEG,09-12-2024,9014.0,9028.0,NaN
CPCMH25-PO-1281280,CPCMH25-PO-1281280,CUT-VEG,10-12-2024,52301.7,55274.2,NaN
...,...,...,...,...,...,...
CPCMH26-PO-2244427,CPCMH26-PO-2244427,CUT-VEG,03-10-2025,7863.0,15268.5,list index out of range;;
CPCMH26-PO-2248894,CPCMH26-PO-2248894,CUT-VEG,04-10-2025,14854.4,26797.0,list index out of range;;
CPCMH26-PO-2252935,CPCMH26-PO-2252935,CUT-VEG,05-10-2025,17686.8,23482.2,list index out of range;;


In [ ]:
mail.logout()

In [ ]:
grnPrDF["PR"] = grnPrDF["PO Amount"] - grnPrDF["GRN"]

In [ ]:
toSave = grnPrDF[["Cut/Veg", "Date", "GRN", "PR", "PO Amount"]]

In [46]:
toSave.index.name = "PO Number"

In [47]:
toSave.to_csv("GRN PR (Blinkit).csv", index=True)